In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 89.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=ccf78fc171b25f497ded3e59bb7213d57225e0d78be86fa0443fdf30391e1474
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# Helper function to generate bits using quantum measurement
def get_quantum_random_bits(n):
    bits = []
    for _ in range(n):
        qc = QuantumCircuit(1, 1)
        qc.h(0) # State: 1/sqrt(2) (|0> + |1>)
        qc.measure(0, 0)
        backend = BasicSimulator()
        job = backend.run(transpile(qc, backend), shots=1)
        bit = int(list(job.result().get_counts().keys())[0])
        bits.append(bit)
    return bits

# --- SETUP ---
N_BITS = 40 # Number of initial qubits
alice_bits = get_quantum_random_bits(N_BITS)
alice_bases = get_quantum_random_bits(N_BITS) # 0 = Z-basis, 1 = X-basis

# --- ALICE ENCODES AND SENDS ---
alice_qubits = []
for i in range(N_BITS):
    qc = QuantumCircuit(1, 1)
    if alice_bits[i] == 1: qc.x(0)
    if alice_bases[i] == 1: qc.h(0) # Prepare in X-basis
    alice_qubits.append(qc)

# --- BOB RECEIVES AND MEASURES ---
bob_bases = get_quantum_random_bits(N_BITS)
bob_results = []

for i in range(N_BITS):
    qc = alice_qubits[i]
    if bob_bases[i] == 1: qc.h(0) # Measure in X-basis
    qc.measure(0, 0)
    backend = BasicSimulator()
    job = backend.run(transpile(qc, backend), shots=1)
    res = int(list(job.result().get_counts().keys())[0])
    bob_results.append(res)

# --- SIFTING PHASE ---
final_alice_key = [alice_bits[i] for i in range(N_BITS) if alice_bases[i] == bob_bases[i]]
final_bob_key = [bob_results[i] for i in range(N_BITS) if alice_bases[i] == bob_bases[i]]

print(f"Alice's Key: {final_alice_key}")
print(f"Bob's Key:   {final_bob_key}")
print(f"Key Match:   {final_alice_key == final_bob_key}")

Alice's Key: [0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1]
Bob's Key:   [0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1]
Key Match:   True
